 Input-Target Pair (X, y)
X = model lai diyeko input/context
y = model le predict garna sikne correct target/output
LLM ma mostly next token prediction ko lagi use huncha

 Example:
 Input (X):  "I love"
 Target (y): "AI"

 Model le X herera y predict garna sikcha

In [1]:
# text lai encode and decode gerney
import re


class Tokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        tokens = text.split()

        return [
            (
                self.str_to_int[token]
                if token in self.str_to_int
                else self.str_to_int["<UNK>"]
            )
            for token in tokens
        ]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return text

In [2]:
# file read garera raw text liyeko
with open(
    r"C:\Users\Acer\Documents\LLMs\LLMs\data\the-verdict.txt", encoding="utf-8"
) as f:
    raw_data = f.read()

In [3]:
preprocess = re.sub(r"[^\w\s]", "", raw_data.lower()).split()

# unique words list banako
all_tokens = sorted(list(set(preprocess)))

# special token for unknown words
all_tokens.append("<UNK>")

vocab = {token: i for i, token in enumerate(all_tokens)}

In [4]:
# tokenizer object banako
tokenizer = Tokenizer(vocab)

# text encode (word to number)
encoded_text = tokenizer.encode(raw_data)

print(len(encoded_text))

3613


In [5]:
# encoded text ko sample print garna 50 oota bahek
enc_sample = all_tokens[50:]
print(enc_sample)

['alone', 'along', 'always', 'amazement', 'amid', 'among', 'amplest', 'amusing', 'an', 'and', 'another', 'answer', 'answered', 'any', 'anything', 'anywhere', 'apparent', 'apparently', 'appearance', 'appeared', 'appointed', 'are', 'arm', 'armchair', 'armchairs', 'arms', 'arrt', 'art', 'articles', 'artist', 'as', 'aside', 'asked', 'at', 'atmosphere', 'atom', 'attack', 'attention', 'attitude', 'audacities', 'away', 'awful', 'axioms', 'azaleas', 'back', 'background', 'balance', 'balancing', 'balustraded', 'basking', 'bathrooms', 'be', 'beaming', 'beanstalk', 'bear', 'beard', 'beardas', 'beauty', 'became', 'because', 'becoming', 'bed', 'been', 'before', 'began', 'begun', 'behind', 'behindbecause', 'being', 'believed', 'beneath', 'bespoke', 'better', 'between', 'big', 'bitsi', 'bitterness', 'blocked', 'born', 'borne', 'boudoir', 'bravura', 'break', 'breaking', 'breathing', 'bricabrac', 'briefly', 'brings', 'bronzes', 'brought', 'brown', 'brush', 'bullthat', 'burlington', 'business', 'but', '

In [6]:
context_size = 5  # model le kati ota past token herera next predict garne

# input context (pahila 5 tokens)
x = enc_sample[:context_size]

# target = next token (context pachhi aune correct word)
y = enc_sample[context_size]

print("input:", x)
print("target:", y)

# input = model lai diyeko past words
# target = model le predict garna sikne next word

input: ['alone', 'along', 'always', 'amazement', 'amid']
target: among


In [7]:
# tokenizer le text lai number ma convert garna sakos vanera encode garna parcha
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    target = enc_sample[i]

    context_text = " ".join(context)
    target_text = target
    print(tokenizer.encode(context_text), "----->:", tokenizer.encode(target_text))

[50] ----->: [51]
[50, 51] ----->: [52]
[50, 51, 52] ----->: [53]
[50, 51, 52, 53] ----->: [54]
[50, 51, 52, 53, 54] ----->: [55]


In [8]:
for i in range(1, context_size + 1):
    # pahilo dekhi i samma ko tokens = input context
    context = enc_sample[:i]

    # context pachhi aune next token = target
    target = enc_sample[i]

    print("input:", context, "----->:", target)

    # model le step-by-step learn garne pattern:
    # yo context deko bela yo word aauchha
    # left ma vako output is input ho aani right ma vako output is target ho

input: ['alone'] ----->: along
input: ['alone', 'along'] ----->: always
input: ['alone', 'along', 'always'] ----->: amazement
input: ['alone', 'along', 'always', 'amazement'] ----->: amid
input: ['alone', 'along', 'always', 'amazement', 'amid'] ----->: among


using Dataset and Dataloader
# Dataset: stores and serves data samples one-by-one
# DataLoader: loads dataset in batches for training efficiently

In [9]:
# pytorch la AI model train garna ko lagi number (TENSOR(array like structure)) structure use garxa
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # text lai token (number) ma convert garne
        token_ids = tokenizer.encode(txt)

        # sliding window use garara training data banaune
        for i in range(0, len(token_ids) - max_length, stride):

            # input sequence (model lai dine data)
            input_chunk = token_ids[i : i + max_length]

            # target sequence (next token predict garna)
            target_chunk = token_ids[i + 1 : i + max_length + 1]

            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)

    def __len__(self):
        # dataset ma kati ota sample cha
        return len(self.input_ids)

    def __getitem__(self, idx):
        # index aanusar input ra target return garne
        return self.input_ids[idx], self.target_ids[idx]

In [10]:
import tiktoken
from torch.utils.data import DataLoader


def create_data_loader(
    txt,
    batch_size=5,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):

    # GPT tokenizer load garne
    tokenizer = tiktoken.get_encoding("gpt2")

    # dataset banaune
    dataset = GPTDataset(txt, tokenizer, max_length, stride)

    # DataLoader le batch ma data dincha
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,  # kati sample ek choti
        shuffle=shuffle,  # data random garne ki nagarne
        drop_last=drop_last,  # last incomplete batch drop garne ki nagarne
        num_workers=num_workers,  # multiprocessing workers
    )

    return dataloader

In [11]:
with open(
    r"C:\Users\Acer\Documents\LLMs\LLMs\data\the-verdict.txt", encoding="utf-8"
) as f:
    raw_data = f.read()

In [13]:
import torch

data_loader = create_data_loader(
    raw_data, batch_size=1, max_length=4, stride=1, shuffle=False
)

# iterator banako
data_iter = iter(data_loader)

# first batch liyeko
first_batch = next(data_iter)

# batch unpack garne (input, target)
x, y = first_batch

print("input ids:", x)
print("target ids:", y)
# output shifted by one

input ids: [tensor([40]), tensor([367]), tensor([2885]), tensor([1464])]
target ids: [tensor([367]), tensor([2885]), tensor([1464]), tensor([1807])]


In [ ]:
import torch

vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# 50257 oota token ko lagi 256 dimension banako
# embedding vector banauna embedding layer initialize garako